# Hands-On 6: Build Human Approval Workflow

We use interrupts to pause a workflow and wait for human approval before execution.

#### Imports

We import `TypedDict` for state, `InMemorySaver` as the checkpointer, and `interrupt`/`Command` for pausing and resuming the graph.

In [1]:
from typing import TypedDict

import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

In [5]:
load_dotenv(r"D:\Agent_Flow\.env", override=True)

print("OpenRouter API key loaded:", bool(os.getenv("OPENROUTER_API_KEY")))

OpenRouter API key loaded: True


In [6]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

print("OpenRouter LLM configured successfully")

OpenRouter LLM configured successfully


In [7]:
response = llm.invoke(
    "Suggest a safe production deployment action for updating an application."
)

print(response.content)

When updating an application in a production environment, it's crucial to follow a safe and systematic approach to minimize downtime and reduce the risk of introducing errors. Here’s a recommended deployment strategy:

### Blue-Green Deployment

**Overview:**
Blue-Green Deployment is a strategy that reduces downtime and risk by running two identical production environments, referred to as "Blue" and "Green." At any time, one environment is live (serving traffic), while the other is idle (ready for the new version).

**Steps:**

1. **Prepare the New Version:**
   - Develop and test the new version of your application in a staging environment that mirrors production.

2. **Set Up the Green Environment:**
   - Deploy the new version of the application to the idle environment (Green) while the current version (Blue) continues to serve traffic.

3. **Run Tests:**
   - Perform smoke tests and other automated tests in the Green environment to ensure that the new version is functioning correct

#### Define State

We define `ApprovalState` with fields for the task, proposed action, approval decision, and the final result.

In [9]:
class ApprovalState(TypedDict):
    task_description: str
    proposed_action: str
    approval: str
    result: str

#### Generate Proposed Action

The plan node simulates an agent proposing an action based on the task description.

In [10]:
def plan_action_node(state: ApprovalState):
    prompt = f"""
You are an AI agent preparing an action that requires human approval.

Task:
{state['task_description']}

Generate one concise proposed action.
Do not execute anything.
Only describe what action should be performed.
"""

    response = llm.invoke(prompt)

    proposed_action = response.content.strip()

    print(f"🤖 Proposed action: {proposed_action}")

    return {
        "proposed_action": proposed_action
    }

In [13]:
def human_approval_node(state: ApprovalState):
    approval = interrupt({
        "question": "Approve this action?",
        "action": state["proposed_action"]
    })

    print(f"🧑 Human decision: {approval}")

    return {
        "approval": approval
    }

In [19]:
def execute_action_node(state: ApprovalState):
    print("🚀 Executing approved action...")

    result = f"Executed: {state['proposed_action']}"

    return {
        "result": result
    }

print("Execute action node created successfully")

Execute action node created successfully


#### Pause Before Execution

The `interrupt()` function pauses the graph and asks the human for input. The graph waits until `Command(resume=...)` is sent.

In [20]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(ApprovalState)

graph.add_node("plan_action", plan_action_node)
graph.add_node("human_approval", human_approval_node)
graph.add_node("execute", execute_action_node)

graph.add_edge(START, "plan_action")
graph.add_edge("plan_action", "human_approval")

graph.add_conditional_edges(
    "human_approval",
    lambda state: (
        "execute"
        if state.get("approval") == "approved"
        else END
    )
)

graph.add_edge("execute", END)

print("Graph created successfully")

Graph created successfully


In [21]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

workflow = graph.compile(
    checkpointer=checkpointer
)

print("Workflow compiled successfully")

Workflow compiled successfully


#### Approve or Reject Action

The execute node only runs if the action was approved. If rejected, the workflow skips execution.

In [22]:
config = {
    "configurable": {
        "thread_id": "approval-1"
    }
}

print("Thread ID:", config["configurable"]["thread_id"])

Thread ID: approval-1


In [23]:
result = workflow.invoke(
    {
        "task_description": "Update production server",
        "proposed_action": "",
        "approval": "",
        "result": ""
    },
    config=config
)

print(result)

🤖 Proposed action: Proposed Action: Update the production server to the latest stable software version to enhance security and performance.
{'task_description': 'Update production server', 'proposed_action': 'Proposed Action: Update the production server to the latest stable software version to enhance security and performance.', 'approval': '', 'result': '', '__interrupt__': [Interrupt(value={'question': 'Approve this action?', 'action': 'Proposed Action: Update the production server to the latest stable software version to enhance security and performance.'}, id='664939eb2456e86ece215aaf9b8bcaa9')]}


In [24]:
snapshot = workflow.get_state(config)

print("Proposed action:")
print(snapshot.values["proposed_action"])

print("\nApproval request:")
print(snapshot.interrupts)

Proposed action:
Proposed Action: Update the production server to the latest stable software version to enhance security and performance.

Approval request:
(Interrupt(value={'question': 'Approve this action?', 'action': 'Proposed Action: Update the production server to the latest stable software version to enhance security and performance.'}, id='664939eb2456e86ece215aaf9b8bcaa9'),)


#### Build Graph with Interrupt

We compile the graph with `interrupt_before=["human_approval"]` so execution pauses before the human node, and route based on the approval decision.

#### Invoke Workflow (Pauses)

When we invoke, the plan node runs, then the graph pauses at the human approval node waiting for input.

In [25]:
from langgraph.types import Command

result = workflow.invoke(
    Command(resume="approved"),
    config=config
)

print("Final result:")
print(result)

🧑 Human decision: approved
🚀 Executing approved action...
Final result:
{'task_description': 'Update production server', 'proposed_action': 'Proposed Action: Update the production server to the latest stable software version to enhance security and performance.', 'approval': 'approved', 'result': 'Executed: Proposed Action: Update the production server to the latest stable software version to enhance security and performance.'}


#### Inspect the Interrupt

Use `get_state()` to see the interrupt payload that is waiting for human input.

#### Resume Graph (Approve)

Use `Command(resume="approved")` to send the human decision and resume the workflow, which then executes the action.

In [ ]:
def human_approval_node(state: ApprovalState):
    approval = interrupt({
        "question": "Approve this action?",
        "action": state["proposed_action"]
    })
    print(f"🧑 Human decision: {approval}")
    return {"approval": approval}

In [26]:
config2 = {
    "configurable": {
        "thread_id": "approval-2"
    }
}

result2 = workflow.invoke(
    {
        "task_description": "Delete production database records",
        "proposed_action": "",
        "approval": "",
        "result": ""
    },
    config=config2
)

print(result2)

🤖 Proposed action: Proposed Action: Permanently delete all records from the production database that are older than one year.
{'task_description': 'Delete production database records', 'proposed_action': 'Proposed Action: Permanently delete all records from the production database that are older than one year.', 'approval': '', 'result': '', '__interrupt__': [Interrupt(value={'question': 'Approve this action?', 'action': 'Proposed Action: Permanently delete all records from the production database that are older than one year.'}, id='488505992058a1d627e5af6e3cc2f544')]}


#### Test: Reject Path

Run a second thread where the human rejects the action, so the execute node is skipped.

In [30]:
result2 = workflow.invoke(
    Command(resume="rejected"),
    config=config2
)

print("Final result:")
print(result2)

Final result:
{'task_description': 'Delete production database records', 'proposed_action': 'Proposed Action: Permanently delete all records from the production database that are older than one year.', 'approval': 'rejected', 'result': ''}


#### Summary

Human-in-the-loop adds safety to agents. The graph pauses at an interrupt, waits for a human decision, and resumes with approval or rejection.